# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srilaya30/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

I first inspect the distributions of important search-performance and content signals. Some metrics can have heavy or skewed distributions, so medians and grouped comparisons are more useful than relying only on averages.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path("/content/flyrank-ml-internship")

if not repo_root.exists():
    !git clone -q https://github.com/Srilaya30/flyrank-ml-internship.git

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# Target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

key_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "avg_position",
    "ctr"
]

display(df[key_fields].describe().T)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

Dataset shape: (30000, 44)


,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.00,27.00,4345.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0



Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal 1 — Impressions

Hypothesis: pages with lower recent impressions may be more likely to have the declining label.

I compare the median recent impressions between the declining and other groups. The result is treated as an observed association, not a causal effect.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal1 = df.groupby("is_declining_label")["impressions_last_30d"].agg(
    ["count", "median", "mean"]
)

display(signal1)

declining_median = signal1.loc[1, "median"]
other_median = signal1.loc[0, "median"]

print("Declining median:", declining_median)
print("Other median:", other_median)

if declining_median < other_median:
    print("Verdict: CONFIRMED")
elif declining_median > other_median:
    print("Verdict: OPPOSITE")
else:
    print("Verdict: MIXED")

,count,median,mean
is_declining_label,,,
0,13738,156.0,2006.126656
1,16262,128.0,941.556635


Declining median: 128.0
Other median: 156.0
Verdict: CONFIRMED


### Signal 2 — Content age

Hypothesis: older content may show a stronger association with the declining label.

I compare content age between the two groups. This tests whether the observed data is consistent with the signal, but it does not show that age causes decline.

In [10]:
signal2 = df.groupby("is_declining_label")["content_age_days"].agg(
    ["count", "median", "mean"]
)

display(signal2)

declining_median = signal2.loc[1, "median"]
other_median = signal2.loc[0, "median"]

print("Declining median:", declining_median)
print("Other median:", other_median)

if declining_median > other_median:
    print("Verdict: CONFIRMED")
elif declining_median < other_median:
    print("Verdict: OPPOSITE")
else:
    print("Verdict: MIXED")

,count,median,mean
is_declining_label,,,
0,13738,287.0,279.829451
1,16262,216.0,236.178637


Declining median: 216.0
Other median: 287.0
Verdict: OPPOSITE


### Signal 3 — CTR

Hypothesis: lower CTR may be associated with the declining label.

I compare CTR distributions between the two groups. This is a directional signal test and not evidence that changing CTR will cause a different outcome.

In [11]:
signal3 = df.groupby("is_declining_label")["ctr"].agg(
    ["count", "median", "mean"]
)

display(signal3)

declining_median = signal3.loc[1, "median"]
other_median = signal3.loc[0, "median"]

print("Declining median:", declining_median)
print("Other median:", other_median)

if declining_median < other_median:
    print("Verdict: CONFIRMED")
elif declining_median > other_median:
    print("Verdict: OPPOSITE")
else:
    print("Verdict: MIXED")

,count,median,mean
is_declining_label,,,
0,13738,0.04,0.731611
1,16262,0.08,0.324138


Declining median: 0.08
Other median: 0.04
Verdict: OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test — Recent impression decline

The action playbook uses recent impression decline as one reason code. I test whether pages with lower recent impressions than the previous period are more frequently associated with the declining label.

The result checks whether the rule's assumption is supported directionally by this dataset.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["impression_decline_flag"] = (
    df["impressions_last_30d"] < df["impressions_prev_30d"]
)

flag_test = df.groupby("impression_decline_flag")["is_declining_label"].agg(
    ["count", "mean", "sum"]
)

display(flag_test)

flag_rate = df.loc[
    df["impression_decline_flag"], "is_declining_label"
].mean()

no_flag_rate = df.loc[
    ~df["impression_decline_flag"], "is_declining_label"
].mean()

print("Declining rate with impression-decline flag:",
      round(flag_rate, 4))

print("Declining rate without flag:",
      round(no_flag_rate, 4))

if flag_rate > no_flag_rate:
    print("Flag-linked verdict: CONFIRMED")
elif flag_rate < no_flag_rate:
    print("Flag-linked verdict: OPPOSITE")
else:
    print("Flag-linked verdict: MIXED")

,count,mean,sum
impression_decline_flag,,,
False,10284,0.000000,0
True,19716,0.824812,16262


Declining rate with impression-decline flag: 0.8248
Declining rate without flag: 0.0
Flag-linked verdict: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal tests provide directional evidence about which search-performance and content signals are associated with the declining label in this dataset. The results can help a content team prioritize pages for human review, but the signals should not be treated as causal explanations. A reviewer should inspect the actual page and business context before taking action.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("W04 SIGNAL AUDIT SUMMARY")
print("=" * 40)

print("Rows analyzed:", len(df))
print("Signals tested: 3")
print("Flag-linked signal: recent impression decline")

print("\nW04 audit completed.")
print("Interpretation: directional decision support only.")

W04 SIGNAL AUDIT SUMMARY
Rows analyzed: 30000
Signals tested: 3
Flag-linked signal: recent impression decline

W04 audit completed.
Interpretation: directional decision support only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [14]:
print("W04 SELF-CHECK")
print("=" * 40)

required_columns = [
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "ctr",
    "is_declining_label"
]

missing_required = [
    c for c in required_columns if c not in df.columns
]

print("Required columns missing:", missing_required)
print("Rows:", len(df))
print("Columns:", len(df.columns))

assert len(missing_required) == 0
assert len(df) > 0

print("\nW04 CHECK: PASS")

W04 SELF-CHECK
Required columns missing: []
Rows: 30000
Columns: 46

W04 CHECK: PASS
